# Article J1 -- Interactive analysis notebook

Runs the same pipeline as `main.py` but stage by stage, so each artifact
(dataframe, model, table, figure) can be inspected in the notebook before
moving to the next stage. See `CLAUDE_Article_J1.md` for the full spec and
`main.py` for the scripted end-to-end run.

Run this from the `article_j1/` directory (or adjust `PROJECT_ROOT` below).

In [ ]:
import os, sys, importlib.util

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
SRC_DIR = os.path.join(PROJECT_ROOT, 'src')
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, SRC_DIR)

import config
from utils import ensure_dirs, print_step
ensure_dirs()

def load(name, filename):
    spec = importlib.util.spec_from_file_location(name, os.path.join(SRC_DIR, filename))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

print('Project root:', PROJECT_ROOT)

## 1-3. Data loading, preprocessing, feature engineering

In [ ]:
dl = load('dl01', '01_data_loading.py')
pp = load('pp02', '02_preprocessing.py')
fe = load('fe03', '03_feature_engineering.py')

data = dl.run()
pp_out = pp.run(data['df'])
fe_out = fe.run(pp_out['df'])
fe_out['df_engineered'].head()

## 4-6. Base ML, Stacking Ensemble, Deep Learning models

In [ ]:
ml = load('ml04', '04_ml_models.py')
stk = load('stk05', '05_stacking_ensemble.py')
dlm = load('dlm06', '06_dl_models.py')

features = fe_out['features']
ml_results, tuning_table = ml.train_all(features, fe_out['X_train_std'], fe_out['y_train'], fe_out['X_test_std'])
stack_result = stk.build_stacking(features, fe_out['X_train_std'], fe_out['y_train'], fe_out['X_test_std'], fe_out['y_test'])
dl_results, group_idx = dlm.train_all(features, fe_out['X_train_mm_dl'], fe_out['y_train_dl'], fe_out['X_val_mm'], fe_out['y_val'], fe_out['X_test_mm'], fe_out['y_test'])
all_results = {**ml_results, 'Stacking Ensemble': stack_result, **dl_results}
list(all_results.keys())

## 7-10. Spatial CV, uncertainty, full metrics (Table 4)

In [ ]:
scv = load('scv08', '08_spatial_cv.py')
unc = load('unc07', '07_uncertainty.py')
ev = load('ev09', '09_evaluation.py')

sloocv_results = scv.run_sloocv(features, fe_out['X_train_std'], fe_out['y_train'], fe_out['coords_train'])
block_cv_table = scv.run_block_cv(features, fe_out['X_train_std'], fe_out['y_train'], fe_out['coords_train'])
uncertainty_results = unc.run(fe_out['X_train_std'], fe_out['y_train'], fe_out['X_test_std'], fe_out['y_test'])
table4 = ev.run(all_results, fe_out['y_test'], features, class_test=fe_out['class_test'], coords_test=fe_out['coords_test'], sloocv_results=sloocv_results)
table4

## 11-13. SHAP, GeoTIFF export, figures & tables

See `main.py` for the full wiring (raster export, all 16 figures, all 8 tables) -- reuse it here via `import runpy; runpy.run_path('../main.py', run_name='__main__')` once the stage-by-stage results above look right, or call the individual `src/11_shap_analysis.py`, `src/10_geotiff_export.py`, `src/12_figures.py`, `src/13_tables.py` functions directly as shown in `main.py`.